# Gene-biotype composition per dataset

This notebook assigns every feature in `gene_total_counts.csv` to a gene biotype from the corresponding species GTF. It retains the original GTF label in `biotype` and adds a harmonized cross-species category in `biotype_group`. For each dataset and biotype it reports:

- total transcript count (the summed UMI count in these count tables);
- percentage of all counts;
- number and percentage of detected features (`total_counts > 0`);
- total number of features from that biotype represented in the count table.

In [11]:
from pathlib import Path
import gzip
import re
import subprocess

import pandas as pd
from IPython.display import display

project_path = Path("/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC")
processed_path = project_path / "data/processed"
output_path = processed_path / "plots/genes/biotypes"
output_path.mkdir(parents=True, exist_ok=True)

## Samples

In [12]:
sample_metadata = [
    {"sample": "SCA013", "sample_type": "Mouse lung",  "technology": "Visium (FFPE)",    "replicate": None, "species": "mouse", "file_path": processed_path / "Visium/SCA013/harpy/gene_total_counts.csv"},
    {"sample": "SCA014", "sample_type": "Mouse liver", "technology": "Visium (FFPE)",    "replicate": None, "species": "mouse", "file_path": processed_path / "Visium/SCA014/harpy/gene_total_counts.csv"},
    {"sample": "SCA016", "sample_type": "Mouse brain", "technology": "Visium (FFPE)",    "replicate": None, "species": "mouse", "file_path": processed_path / "Visium/SCA016/harpy/gene_total_counts.csv"},
    {"sample": "SCA015", "sample_type": "Human lung",  "technology": "Visium (FFPE)",    "replicate": None, "species": "human", "file_path": processed_path / "Visium/SCA015/harpy/gene_total_counts.csv"},

    {"sample": "SCA001", "sample_type": "Mouse lung",  "technology": "Visium HD (FFPE)", "replicate": None, "species": "mouse", "file_path": processed_path / "Visium_HD/SCA001/harpy/gene_total_counts.csv"},
    {"sample": "SCA002", "sample_type": "Mouse liver", "technology": "Visium HD (FFPE)", "replicate": None, "species": "mouse", "file_path": processed_path / "Visium_HD/SCA002/harpy/gene_total_counts.csv"},
    {"sample": "SCA003", "sample_type": "Mouse brain", "technology": "Visium HD (FFPE)", "replicate": None, "species": "mouse", "file_path": processed_path / "Visium_HD/SCA003/harpy/gene_total_counts.csv"},
    {"sample": "SCA010", "sample_type": "Human lung",  "technology": "Visium HD (FFPE)", "replicate": None, "species": "human", "file_path": processed_path / "Visium_HD/SCA010/harpy/gene_total_counts.csv"},

    {"sample": "SCA017", "sample_type": "Mouse lung",  "technology": "Visium HD (FF)",   "replicate": None, "species": "mouse", "file_path": processed_path / "Visium_HD/SCA017/harpy/gene_total_counts.csv"},
    {"sample": "SCA018", "sample_type": "Mouse liver", "technology": "Visium HD (FF)",   "replicate": None, "species": "mouse", "file_path": processed_path / "Visium_HD/SCA018/harpy/gene_total_counts.csv"},
    {"sample": "SCA019", "sample_type": "Mouse brain", "technology": "Visium HD (FF)",   "replicate": None, "species": "mouse", "file_path": processed_path / "Visium_HD/SCA019/harpy/gene_total_counts.csv"},
    {"sample": "SCA020", "sample_type": "Human lung",  "technology": "Visium HD (FF)",   "replicate": None, "species": "human", "file_path": processed_path / "Visium_HD/SCA020/harpy/gene_total_counts.csv"},

    {"sample": "SCA021", "sample_type": "Mouse liver", "technology": "Visium HD 3'",     "replicate": 1,    "species": "mouse", "file_path": processed_path / "Visium_HD/SCA021/harpy/gene_total_counts.csv"},
    {"sample": "SCA023", "sample_type": "Mouse liver", "technology": "Visium HD 3'",     "replicate": 2,    "species": "mouse", "file_path": processed_path / "Visium_HD/SCA023/harpy/gene_total_counts.csv"},
    {"sample": "SCA024", "sample_type": "Mouse brain", "technology": "Visium HD 3'",     "replicate": None, "species": "mouse", "file_path": processed_path / "Visium_HD/SCA024/harpy/gene_total_counts.csv"},
    {"sample": "FPE010", "sample_type": "Rice root",   "technology": "Visium HD 3'",     "replicate": None, "species": "rice",  "file_path": processed_path / "Visium_HD/FPE010/harpy/gene_total_counts.csv"},

    {"sample": "Exp74",  "sample_type": "Mouse brain", "technology": "BMKMANU S1000",    "replicate": None, "species": "mouse", "file_path": processed_path / "BMK_S1000/exp74/harpy/gene_total_counts.csv"},
    {"sample": "Exp101", "sample_type": "Mouse lung",  "technology": "BMKMANU S3000",    "replicate": None, "species": "mouse", "file_path": processed_path / "BMK_S3000/Exp101/harpy/gene_total_counts.csv"},
    {"sample": "Exp100", "sample_type": "Mouse liver", "technology": "BMKMANU S3000",    "replicate": None, "species": "mouse", "file_path": processed_path / "BMK_S3000/Exp100/harpy/gene_total_counts.csv"},
    {"sample": "Exp99",  "sample_type": "Mouse brain", "technology": "BMKMANU S3000",    "replicate": None, "species": "mouse", "file_path": processed_path / "BMK_S3000/Exp99/harpy/gene_total_counts.csv"},
    {"sample": "Exp102", "sample_type": "Human lung",  "technology": "BMKMANU S3000",    "replicate": None, "species": "human", "file_path": processed_path / "BMK_S3000/Exp102/harpy/gene_total_counts.csv"},
    {"sample": "psb",    "sample_type": "Rice root",   "technology": "BMKMANU S3000",    "replicate": None, "species": "rice",  "file_path": processed_path / "BMK_S3000/psb/harpy/gene_total_counts.csv"},

    {"sample": "SC12", "sample_type": "Mouse lung",  "technology": "Curio Seeker", "replicate": None, "species": "mouse", "file_path": processed_path / "Curio_seeker/SC12/harpy/gene_total_counts.csv"},
    {"sample": "SC13", "sample_type": "Mouse liver", "technology": "Curio Seeker", "replicate": None, "species": "mouse", "file_path": processed_path / "Curio_seeker/SC13/harpy/gene_total_counts.csv"},
    {"sample": "SC11", "sample_type": "Mouse brain", "technology": "Curio Seeker", "replicate": None, "species": "mouse", "file_path": processed_path / "Curio_seeker/SC11/harpy/gene_total_counts.csv"},
    {"sample": "SC14", "sample_type": "Human lung",  "technology": "Curio Seeker", "replicate": None, "species": "human", "file_path": processed_path / "Curio_seeker/SC14/harpy/gene_total_counts.csv"},

    {"sample": "Exp93-sampleSPC002",  "sample_type": "Mouse lung",  "technology": "Nova-ST", "replicate": None, "species": "mouse", "file_path": processed_path / "Nova-ST/Exp93-sampleSPC002/harpy/gene_total_counts.csv"},
    {"sample": "Exp100-sampleSPC004", "sample_type": "Mouse liver", "technology": "Nova-ST", "replicate": None, "species": "mouse", "file_path": processed_path / "Nova-ST/Exp100-sampleSPC004/harpy/gene_total_counts.csv"},
    {"sample": "Exp65-sampleSPC014",  "sample_type": "Mouse brain", "technology": "Nova-ST", "replicate": None, "species": "mouse", "file_path": processed_path / "Nova-ST/Exp65-sampleSPC014/harpy/gene_total_counts.csv"},
    {"sample": "Exp93-sampleSPC022",  "sample_type": "Human lung",  "technology": "Nova-ST", "replicate": None, "species": "human", "file_path": processed_path / "Nova-ST/Exp93-sampleSPC022/harpy/gene_total_counts.csv"},
    {"sample": "Exp100-samplePSB",    "sample_type": "Rice root",   "technology": "Nova-ST", "replicate": None, "species": "rice",  "file_path": processed_path / "Nova-ST/Exp100-samplePSB/harpy/gene_total_counts.csv"},

    {"sample": "spc002",  "sample_type": "Mouse lung",  "technology": "Stereo-Seq", "replicate": None, "species": "mouse", "file_path": processed_path / "Stereo-Seq/spc002/harpy/gene_total_counts.csv"},
    {"sample": "spc004",  "sample_type": "Mouse liver", "technology": "Stereo-Seq", "replicate": None, "species": "mouse", "file_path": processed_path / "Stereo-Seq/spc004/harpy/gene_total_counts.csv"},
    {"sample": "spc014",  "sample_type": "Mouse brain", "technology": "Stereo-Seq", "replicate": None, "species": "mouse", "file_path": processed_path / "Stereo-Seq/spc014/harpy/gene_total_counts.csv"},
    {"sample": "spc022",  "sample_type": "Human lung",  "technology": "Stereo-Seq", "replicate": None, "species": "human", "file_path": processed_path / "Stereo-Seq/spc022/harpy/gene_total_counts.csv"},
    {"sample": "psb_rice", "sample_type": "Rice root",   "technology": "Stereo-Seq", "replicate": None, "species": "rice",  "file_path": processed_path / "Stereo-Seq/psb/harpy/gene_total_counts.csv"},
]

missing_files = [str(s["file_path"]) for s in sample_metadata if not s["file_path"].exists()]
print(f"Samples: {len(sample_metadata)}; missing count files: {len(missing_files)}")
missing_files[:10]

Samples: 36; missing count files: 0


[]

## Gene annotations

The mouse, human, and rice datasets are annotated against the matching reference GTFs. Only `gene` records are used. The parser accepts either `gene_type` (mouse/human GENCODE references) or `gene_biotype` (rice Ensembl Plants reference). Original labels are retained and mapped to broader Ensembl/GENCODE-based groups for cross-species comparison. Parsed annotations are cached as small compressed CSV files, so the large GTFs only need to be scanned once.

In [13]:
gtf_paths = {
    "mouse": Path("/data/groups/technologies/spatial.catalyst/Julien/spaceranger/references/refdata-gex-mm10-2020-A/genes/genes.gtf"),
    "human": Path("/data/groups/technologies/spatial.catalyst/Julien/spaceranger/references/refdata-gex-GRCh38-2020-A/genes/genes.gtf"),
    "rice": Path("/data/groups/technologies/spatial.catalyst/Julien/spaceranger/references/IRGSP.49/genes/genes.gtf"),
}

for species, path in gtf_paths.items():
    print(f"{species:>5}: {'found' if path.exists() else 'MISSING'}  {path}")

mouse: found  /data/groups/technologies/spatial.catalyst/Julien/spaceranger/references/refdata-gex-mm10-2020-A/genes/genes.gtf
human: found  /data/groups/technologies/spatial.catalyst/Julien/spaceranger/references/refdata-gex-GRCh38-2020-A/genes/genes.gtf
 rice: found  /data/groups/technologies/spatial.catalyst/Julien/spaceranger/references/IRGSP.49/genes/genes.gtf


In [14]:
attribute_pattern = re.compile(r'(\S+) \"([^\"]*)\"')

def clean_gene_id(values):
    # Remove an optional Ensembl version suffix, e.g. ENSG...12.5 -> ENSG...12.
    return values.astype('string').str.replace(r'\.\d+$', '', regex=True)

def read_gene_annotations(gtf_path, cache_path):
    if cache_path.exists():
        return pd.read_csv(cache_path, dtype='string')

    opener = gzip.open if str(gtf_path).endswith('.gz') else open
    records = []
    with opener(gtf_path, 'rt') as handle:
        for line in handle:
            if line.startswith('#'):
                continue
            fields = line.rstrip('\n').split('\t', 8)
            if len(fields) != 9 or fields[2] != 'gene':
                continue
            attributes = dict(attribute_pattern.findall(fields[8]))
            gene_id = attributes.get('gene_id')
            if gene_id is None:
                continue
            records.append({
                'gene_id': gene_id.split('.')[0],
                'gene_name': attributes.get('gene_name'),
                'biotype': attributes.get('gene_type', attributes.get('gene_biotype', 'unannotated')),
            })

    annotations = pd.DataFrame(records).drop_duplicates('gene_id')
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    annotations.to_csv(cache_path, index=False, compression='gzip')
    return annotations

annotation_cache = output_path / 'annotation_cache'
gene_annotations = {
    species: read_gene_annotations(path, annotation_cache / f'{species}_gene_biotypes.csv.gz')
    for species, path in gtf_paths.items()
}

def harmonize_biotype(biotype):
    # Broad analysis groups based on Ensembl/GENCODE biotype classes.
    if pd.isna(biotype) or biotype == 'unannotated':
        return 'Unannotated'
    if biotype == 'tRNA_pseudogene':
        return 'Other ncRNA'
    if 'pseudogene' in biotype:
        return 'Pseudogene'
    if biotype.startswith('IG_'):
        return 'IG genes'
    if biotype.startswith('TR_'):
        return 'TR genes'
    if biotype in {'protein_coding', 'nontranslating_CDS'}:
        return 'Protein-coding'
    if biotype in {
        'lncRNA', 'lincRNA', 'antisense', 'antisense_RNA',
        'sense_intronic', 'sense_overlapping',
    }:
        return 'lncRNA'
    if biotype in {'rRNA', 'Mt_rRNA'}:
        return 'rRNA'
    if biotype in {'tRNA', 'Mt_tRNA'}:
        return 'tRNA'
    if biotype in {
        'snoRNA', 'snRNA', 'miRNA', 'pre_miRNA',
        'SRP_RNA', 'RNase_MRP_RNA',
    }:
        return 'Other ncRNA'
    return 'Other'

for annotation in gene_annotations.values():
    annotation['biotype_group'] = annotation['biotype'].map(harmonize_biotype)

# Use gene names only as a fallback and only when one name maps to one biotype.
gene_name_biotypes = {
    species: (
        annotation.dropna(subset=['gene_name', 'biotype'])
        .groupby('gene_name')['biotype']
        .agg(lambda x: x.iloc[0] if x.nunique() == 1 else pd.NA)
        .dropna()
    )
    for species, annotation in gene_annotations.items()
}

for species, annotation in gene_annotations.items():
    print(f"{species:>5}: {len(annotation):,} annotated genes; {annotation['biotype'].nunique():,} biotypes")

mouse: 32,285 annotated genes; 15 biotypes
human: 36,601 annotated genes; 15 biotypes
 rice: 38,978 annotated genes; 13 biotypes


### Biotype content of the processing references

This inventory counts the unique gene IDs assigned to each biotype in the exact processed GTF used for each species. It distinguishes a biotype that was available for quantification from one that was absent from the reference and therefore could not be detected.

In [15]:
reference_biotype_summary = pd.concat(
    [
        (
            annotation.groupby(['biotype', 'biotype_group'], dropna=False)
            .size()
            .rename('number_of_annotated_genes')
            .reset_index()
            .assign(species=species, reference_gtf=str(gtf_paths[species]))
        )
        for species, annotation in gene_annotations.items()
    ],
    ignore_index=True,
)
reference_biotype_summary['percentage_of_reference_genes'] = (
    100
    * reference_biotype_summary['number_of_annotated_genes']
    / reference_biotype_summary.groupby('species')['number_of_annotated_genes'].transform('sum')
)
reference_biotype_summary = reference_biotype_summary[[
    'species', 'reference_gtf', 'biotype', 'biotype_group',
    'number_of_annotated_genes', 'percentage_of_reference_genes',
]].sort_values(['species', 'number_of_annotated_genes'], ascending=[True, False])

reference_biotype_summary.to_csv(
    output_path / 'reference_gene_biotype_inventory.csv', index=False
)

reference_biotype_counts = (
    reference_biotype_summary
    .pivot(index='biotype', columns='species', values='number_of_annotated_genes')
    .fillna(0)
    .astype(int)
)
display(reference_biotype_counts)

reference_biotype_group_counts = (
    reference_biotype_summary
    .groupby(['biotype_group', 'species'])['number_of_annotated_genes']
    .sum()
    .unstack(fill_value=0)
)
display(reference_biotype_group_counts)

print('rRNA/tRNA annotations in the exact processing references:')
display(
    reference_biotype_summary.loc[
        reference_biotype_summary['biotype'].str.contains('rRNA|tRNA', case=False, na=False)
    ]
)

species,human,mouse,rice
biotype,,,
IG_C_gene,14,13,0
IG_C_pseudogene,9,1,0
IG_D_gene,37,19,0
IG_J_gene,18,14,0
IG_J_pseudogene,3,0,0
IG_LV_gene,0,4,0
IG_V_gene,144,218,0
IG_V_pseudogene,188,158,0
RNase_MRP_RNA,0,0,1


species,human,mouse,rice
biotype_group,,,
IG genes,213,268,0
Other ncRNA,0,0,405
Protein-coding,19394,21700,37960
Pseudogene,237,203,1
TR genes,195,226,0
lncRNA,16562,9888,95
rRNA,0,0,275
tRNA,0,0,242


rRNA/tRNA annotations in the exact processing references:


,species,reference_gtf,biotype,biotype_group,number_of_annotated_genes,percentage_of_reference_genes
37,rice,/data/groups/technologies/spatial.catalyst/Jul...,rRNA,rRNA,275,0.705526
41,rice,/data/groups/technologies/spatial.catalyst/Jul...,tRNA,tRNA,242,0.620863
42,rice,/data/groups/technologies/spatial.catalyst/Jul...,tRNA_pseudogene,Other ncRNA,6,0.015393


## Summarize each dataset

Gene IDs are matched first. For an unmatched ID, the notebook falls back to gene name only when that name has one unambiguous biotype in the GTF. Remaining features are retained as `unannotated`.

In [16]:
def summarize_biotypes(sample, annotations, gene_name_biotypes):
    # Curio Seeker tables contain gene symbols but no gene_ID column.
    # Use IDs when available and otherwise rely on the unambiguous-name map.
    columns = pd.read_csv(sample['file_path'], nrows=0).columns
    usecols = ['gene', 'total_counts'] + (['gene_ID'] if 'gene_ID' in columns else [])
    counts = pd.read_csv(
        sample['file_path'],
        usecols=usecols,
        dtype={column: 'string' for column in ['gene', 'gene_ID'] if column in usecols},
    )
    if 'gene_ID' not in counts:
        counts['gene_ID'] = pd.Series(pd.NA, index=counts.index, dtype='string')
    counts['total_counts'] = pd.to_numeric(counts['total_counts'], errors='raise').fillna(0)
    counts['gene_id_clean'] = clean_gene_id(counts['gene_ID'])

    annotated = counts.merge(
        annotations[['gene_id', 'biotype']],
        how='left',
        left_on='gene_id_clean',
        right_on='gene_id',
    )

    annotated['biotype'] = annotated['biotype'].fillna(
        annotated['gene'].map(gene_name_biotypes)
    )
    annotated['biotype'] = annotated['biotype'].fillna('unannotated')
    annotated['biotype_group'] = annotated['biotype'].map(harmonize_biotype)

    total_counts = annotated['total_counts'].sum()
    total_detected = int((annotated['total_counts'] > 0).sum())

    summary = (
        annotated.groupby(['biotype', 'biotype_group'], as_index=False, observed=True)
        .agg(
            total_transcript_count=('total_counts', 'sum'),
            number_of_features_in_count_table=('gene', 'size'),
            number_of_detected_features=('total_counts', lambda x: int((x > 0).sum())),
        )
    )
    summary['percentage_of_counts'] = 100 * summary['total_transcript_count'] / total_counts
    summary['percentage_of_detected_features'] = (
        100 * summary['number_of_detected_features'] / total_detected
    )
    summary['percentage_of_biotype_features_detected'] = (
        100 * summary['number_of_detected_features'] / summary['number_of_features_in_count_table']
    )

    metadata = {
        'sample': sample['sample'],
        'sample_type': sample['sample_type'],
        'technology': sample['technology'],
        'replicate': sample['replicate'],
        'species': sample['species'],
    }
    summary = summary.assign(**metadata)
    summary = summary[[
        'sample', 'sample_type', 'technology', 'replicate', 'species',
        'biotype', 'biotype_group',
        'total_transcript_count', 'percentage_of_counts',
        'number_of_detected_features', 'percentage_of_detected_features',
        'number_of_features_in_count_table', 'percentage_of_biotype_features_detected',
    ]].sort_values('total_transcript_count', ascending=False)

    unannotated = annotated['biotype'].eq('unannotated')
    qc = {
        **metadata,
        'number_of_features': len(annotated),
        'number_of_unannotated_features': int(unannotated.sum()),
        'percentage_of_features_annotated': 100 * (~unannotated).mean(),
        'percentage_of_counts_annotated': 100 * annotated.loc[~unannotated, 'total_counts'].sum() / total_counts,
        'total_transcript_count': total_counts,
    }
    return summary, qc, annotated[[
        'gene', 'gene_ID', 'total_counts', 'biotype', 'biotype_group',
    ]].copy()

all_summaries = []
mapping_qc = []
annotated_datasets = {}

for sample in sample_metadata:
    summary, qc, annotated = summarize_biotypes(
        sample,
        gene_annotations[sample['species']],
        gene_name_biotypes[sample['species']],
    )
    all_summaries.append(summary)
    mapping_qc.append(qc)
    annotated_datasets[(sample['technology'], sample['sample'])] = annotated

biotype_summary = pd.concat(all_summaries, ignore_index=True)
mapping_qc = pd.DataFrame(mapping_qc)

biotype_summary.to_csv(output_path / 'gene_biotype_summary_all_datasets.csv', index=False)

print(f"Saved the combined biotype table to {output_path}")

Saved the combined biotype table to /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/plots/genes/biotypes


In [17]:
display(mapping_qc)
display(biotype_summary.head(30))

,sample,sample_type,technology,replicate,species,number_of_features,number_of_unannotated_features,percentage_of_features_annotated,percentage_of_counts_annotated,total_transcript_count
0,SCA013,Mouse lung,Visium (FFPE),NaN,mouse,19032,0,100.000000,100.000000,44648812.0
1,SCA014,Mouse liver,Visium (FFPE),NaN,mouse,19028,0,100.000000,100.000000,549766433.0
2,SCA016,Mouse brain,Visium (FFPE),NaN,mouse,19008,0,100.000000,100.000000,75262315.0
3,SCA015,Human lung,Visium (FFPE),NaN,human,17991,0,100.000000,100.000000,17935541.0
4,SCA001,Mouse lung,Visium HD (FFPE),NaN,mouse,19032,0,100.000000,100.000000,64515249.0
5,SCA002,Mouse liver,Visium HD (FFPE),NaN,mouse,18999,0,100.000000,100.000000,413309287.0
6,SCA003,Mouse brain,Visium HD (FFPE),NaN,mouse,19011,0,100.000000,100.000000,216805241.0
7,SCA010,Human lung,Visium HD (FFPE),NaN,human,18018,0,100.000000,100.000000,39873094.0
8,SCA017,Mouse lung,Visium HD (FF),NaN,mouse,17636,0,100.000000,100.000000,99960007.0
9,SCA018,Mouse liver,Visium HD (FF),NaN,mouse,16104,0,100.000000,100.000000,561663318.0


,sample,sample_type,technology,replicate,species,biotype,biotype_group,total_transcript_count,percentage_of_counts,number_of_detected_features,percentage_of_detected_features,number_of_features_in_count_table,percentage_of_biotype_features_detected
0,SCA013,Mouse lung,Visium (FFPE),None,mouse,protein_coding,Protein-coding,44591174.0,99.870908,18964,99.642707,18964,100.0
1,SCA013,Mouse lung,Visium (FFPE),None,mouse,IG_C_gene,IG genes,41240.0,0.092365,12,0.063052,12,100.0
2,SCA013,Mouse lung,Visium (FFPE),None,mouse,lncRNA,lncRNA,6094.0,0.013649,1,0.005254,1,100.0
3,SCA013,Mouse lung,Visium (FFPE),None,mouse,TR_V_gene,TR genes,4789.0,0.010726,26,0.136612,26,100.0
4,SCA013,Mouse lung,Visium (FFPE),None,mouse,TR_C_gene,TR genes,2965.0,0.006641,7,0.036780,7,100.0
5,SCA013,Mouse lung,Visium (FFPE),None,mouse,IG_V_gene,IG genes,2550.0,0.005711,22,0.115595,22,100.0
6,SCA014,Mouse liver,Visium (FFPE),None,mouse,protein_coding,Protein-coding,549718078.0,99.991204,18959,99.637376,18959,100.0
7,SCA014,Mouse liver,Visium (FFPE),None,mouse,IG_C_gene,IG genes,31537.0,0.005736,12,0.063065,12,100.0
8,SCA014,Mouse liver,Visium (FFPE),None,mouse,lncRNA,lncRNA,9726.0,0.001769,1,0.005255,1,100.0
9,SCA014,Mouse liver,Visium (FFPE),None,mouse,TR_V_gene,TR genes,3123.0,0.000568,27,0.141896,27,100.0


## Biotype composition of overlapping genes
Within each tissue type, a gene is **shared across all datasets** when it has `total_counts > 0` in every included dataset. Overlap is calculated separately for each tissue type, so only biologically matched samples are compared.

Two complementary results are saved:

1. **Feature composition:** the number and percentage of shared genes assigned to each biotype. This is one result per tissue type because all datasets share the same gene set by definition.
2. **Count composition:** for every dataset, the UMI counts contributed by each biotype among the shared genes. This can differ between datasets even though the identities of the shared genes are fixed.

For comparison, each per-dataset table also contains genes detected in that dataset that are *not* shared across all datasets. This is a convenient comparison group, but it should not be interpreted as genes unique to that dataset.

In [18]:
def resolve_biotype(values):
    unique = pd.unique(values.dropna())
    if len(unique) == 1:
        return unique[0]
    if len(unique) == 0:
        return 'unannotated'
    return 'ambiguous'

def collapse_to_gene_symbols(frame):
    # genes.R defines overlap by gene symbol. Collapse any repeated symbols first.
    return (
        frame.dropna(subset=['gene'])
        .groupby('gene', as_index=False, observed=True)
        .agg(
            total_counts=('total_counts', 'sum'),
            biotype=('biotype', resolve_biotype),
            biotype_group=('biotype_group', resolve_biotype),
        )
    )

overlap_feature_summaries = []
overlap_dataset_summaries = []

for sample_type in sorted({sample['sample_type'] for sample in sample_metadata}):
    tissue_samples = [sample for sample in sample_metadata if sample['sample_type'] == sample_type]
    detected_tables = {}

    for sample in tissue_samples:
        key = (sample['technology'], sample['sample'])
        table = collapse_to_gene_symbols(annotated_datasets[key])
        detected_tables[key] = table.loc[table['total_counts'] > 0].copy()

    common_genes = set.intersection(
        *(set(table['gene']) for table in detected_tables.values())
    )

    # Resolve one consistent biotype per gene across all datasets of this tissue.
    tissue_biotypes = (
        pd.concat(
            [table[['gene', 'biotype']] for table in detected_tables.values()],
            ignore_index=True,
        )
        .groupby('gene', observed=True)['biotype']
        .agg(resolve_biotype)
    )
    tissue_biotype_groups = (
        pd.concat(
            [table[['gene', 'biotype_group']] for table in detected_tables.values()],
            ignore_index=True,
        )
        .groupby('gene', observed=True)['biotype_group']
        .agg(resolve_biotype)
    )

    shared = pd.DataFrame({'gene': sorted(common_genes)})
    shared['biotype'] = shared['gene'].map(tissue_biotypes).fillna('unannotated')
    shared['biotype_group'] = (
        shared['gene'].map(tissue_biotype_groups).fillna('Unannotated')
    )
    shared.insert(0, 'sample_type', sample_type)
    feature_summary = (
        shared.groupby(['biotype', 'biotype_group'], as_index=False, observed=True)
        .agg(number_of_shared_features=('gene', 'size'))
    )
    feature_summary['percentage_of_shared_features'] = (
        100 * feature_summary['number_of_shared_features'] / len(shared)
    )
    feature_summary.insert(0, 'number_of_datasets', len(tissue_samples))
    feature_summary.insert(0, 'sample_type', sample_type)
    overlap_feature_summaries.append(feature_summary)

    for sample in tissue_samples:
        key = (sample['technology'], sample['sample'])
        frame = detected_tables[key].copy()
        frame['biotype'] = frame['gene'].map(tissue_biotypes).fillna('unannotated')
        frame['biotype_group'] = (
            frame['gene'].map(tissue_biotype_groups).fillna('Unannotated')
        )
        frame['gene_set'] = 'not_shared_across_all_datasets'
        frame.loc[frame['gene'].isin(common_genes), 'gene_set'] = 'shared_across_all_datasets'

        summary = (
            frame.groupby(
                ['gene_set', 'biotype', 'biotype_group'],
                as_index=False,
                observed=True,
            )
            .agg(
                total_umi_count=('total_counts', 'sum'),
                number_of_detected_features=('gene', 'size'),
            )
        )
        summary['percentage_of_counts_within_gene_set'] = (
            100 * summary['total_umi_count']
            / summary.groupby('gene_set')['total_umi_count'].transform('sum')
        )
        summary['percentage_of_features_within_gene_set'] = (
            100 * summary['number_of_detected_features']
            / summary.groupby('gene_set')['number_of_detected_features'].transform('sum')
        )
        summary['percentage_of_all_dataset_counts'] = (
            100 * summary['total_umi_count'] / frame['total_counts'].sum()
        )
        summary['percentage_of_all_detected_features'] = (
            100 * summary['number_of_detected_features'] / len(frame)
        )
        summary.insert(0, 'species', sample['species'])
        summary.insert(0, 'replicate', sample['replicate'])
        summary.insert(0, 'technology', sample['technology'])
        summary.insert(0, 'sample', sample['sample'])
        summary.insert(0, 'sample_type', sample_type)
        overlap_dataset_summaries.append(summary)

overlap_feature_summary = pd.concat(overlap_feature_summaries, ignore_index=True)
overlap_dataset_summary = pd.concat(overlap_dataset_summaries, ignore_index=True)
overlap_feature_summary.to_csv(
    output_path / 'shared_gene_biotype_composition_by_tissue.csv', index=False
)
overlap_dataset_summary.to_csv(
    output_path / 'overlap_biotype_composition_all_datasets.csv', index=False
)

print(f"Saved the two combined overlap tables to {output_path}")
print(
    overlap_feature_summary.groupby('sample_type')['number_of_shared_features']
    .sum()
    .rename('number_of_genes_shared_across_all_datasets')
)

Saved the two combined overlap tables to /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/plots/genes/biotypes
sample_type
Human lung     14325
Mouse brain    14263
Mouse liver    12731
Mouse lung     14315
Rice root      18462
Name: number_of_genes_shared_across_all_datasets, dtype: int64


In [19]:
display(overlap_feature_summary)
display(overlap_dataset_summary.head(30))

,sample_type,number_of_datasets,biotype,biotype_group,number_of_shared_features,percentage_of_shared_features
0,Human lung,7,IG_C_gene,IG genes,10,0.069808
1,Human lung,7,TR_C_gene,TR genes,6,0.041885
2,Human lung,7,ambiguous,ambiguous,1,0.006981
3,Human lung,7,lncRNA,lncRNA,8,0.055846
4,Human lung,7,protein_coding,Protein-coding,14300,99.825480
5,Mouse brain,9,IG_C_gene,IG genes,6,0.042067
6,Mouse brain,9,TR_C_gene,TR genes,3,0.021033
7,Mouse brain,9,lncRNA,lncRNA,1,0.007011
8,Mouse brain,9,protein_coding,Protein-coding,14253,99.929889
9,Mouse liver,9,IG_C_gene,IG genes,7,0.054984


,sample_type,sample,technology,replicate,species,gene_set,biotype,biotype_group,total_umi_count,number_of_detected_features,percentage_of_counts_within_gene_set,percentage_of_features_within_gene_set,percentage_of_all_dataset_counts,percentage_of_all_detected_features
0,Human lung,SCA015,Visium (FFPE),None,human,not_shared_across_all_datasets,TR_J_gene,TR genes,76.0,2,0.007813,0.054555,0.000424,0.011117
1,Human lung,SCA015,Visium (FFPE),None,human,not_shared_across_all_datasets,lncRNA,lncRNA,999.0,5,0.102697,0.136388,0.005570,0.027792
2,Human lung,SCA015,Visium (FFPE),None,human,not_shared_across_all_datasets,protein_coding,Protein-coding,971687.0,3659,99.889490,99.809056,5.417662,20.337947
3,Human lung,SCA015,Visium (FFPE),None,human,shared_across_all_datasets,IG_C_gene,IG genes,244836.0,10,1.443372,0.069808,1.365088,0.055583
4,Human lung,SCA015,Visium (FFPE),None,human,shared_across_all_datasets,TR_C_gene,TR genes,3056.0,6,0.018016,0.041885,0.017039,0.033350
5,Human lung,SCA015,Visium (FFPE),None,human,shared_across_all_datasets,ambiguous,ambiguous,690.0,1,0.004068,0.006981,0.003847,0.005558
6,Human lung,SCA015,Visium (FFPE),None,human,shared_across_all_datasets,lncRNA,lncRNA,5253.0,8,0.030968,0.055846,0.029288,0.044467
7,Human lung,SCA015,Visium (FFPE),None,human,shared_across_all_datasets,protein_coding,Protein-coding,16708944.0,14300,98.503577,99.825480,93.161082,79.484187
8,Human lung,SCA010,Visium HD (FFPE),None,human,not_shared_across_all_datasets,TR_J_gene,TR genes,268.0,2,0.020180,0.054157,0.000672,0.011100
9,Human lung,SCA010,Visium HD (FFPE),None,human,not_shared_across_all_datasets,lncRNA,lncRNA,788.0,5,0.059336,0.135391,0.001976,0.027750
